
## Install Library yang Dibutuhkan

In [1]:



!pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.5 MB/s eta 0:00:00


## Import Library dan Mount Google Drive


In [2]:


import os
import re
import string
from google.colab import drive
from pdfminer.high_level import extract_text

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


# Folder Utama
BASE_DRIVE_DIR = '/content/drive/MyDrive/CBR/'

# Mensuaikan direktori output
RAW_DIR = os.path.join(BASE_DRIVE_DIR, 'data/raw/')
LOG_DIR = os.path.join(BASE_DRIVE_DIR, 'logs/')

# File putusan
SOURCE_PDF_DIR = os.path.join(BASE_DRIVE_DIR, 'putusan_pdf/')


os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SOURCE_PDF_DIR, exist_ok=True)

print(f"\nStruktur direktori di Google Drive berhasil disiapkan:")
print(f"- Folder Taruh PDF Mentah : {SOURCE_PDF_DIR}")
print(f"- Folder Output Teks Bersih: {RAW_DIR}")
print(f"- Folder Log Bersih        : {LOG_DIR}")



Struktur direktori di Google Drive berhasil disiapkan:
- Folder Taruh PDF Mentah : /content/drive/MyDrive/CBR/putusan_pdf/
- Folder Output Teks Bersih: /content/drive/MyDrive/CBR/data/raw/
- Folder Log Bersih        : /content/drive/MyDrive/CBR/logs/


## Fungsi Pembersihan Teks (Cleaning)

In [4]:


def clean_indonesian_court_text(text):
    if not text:
        return ""

    # Hapus pola header/footer Direktori Putusan Mahkamah Agung
    text = re.sub(re.compile(r'direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia', re.IGNORECASE), '', text)
    text = re.sub(re.compile(r'Disclaimer\s+.*', re.IGNORECASE), '', text)
    text = re.sub(re.compile(r'halaman\s+\d+\s+dari\s+\d+', re.IGNORECASE), '', text)
    text = re.sub(re.compile(r'\bputusan\s+nomor\s+.*\b', re.IGNORECASE), '', text)

    # Normalisasi Karakter (Lower-case)
    text = text.lower()

    # Normalisasi spasi dan baris baru
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text


## Pipeline Utama: Proses Semua PDF

In [5]:


log_file_path = os.path.join(LOG_DIR, 'cleaning.log')

# Mengecek apakah file PDF sudah diunggah ke Drive
pdf_files = [f for f in os.listdir(SOURCE_PDF_DIR) if f.lower().endswith('.pdf')]

if len(pdf_files) == 0:
    print(f"[PERINGATAN] Folder '{SOURCE_PDF_DIR}' di Drivemasih kosong!")
    print("Silakan buka Google Drive, dan masukkan folder.")
else:
    print(f"Menemukan {len(pdf_files)} file PDF di Google Drive. Memulai proses...\n" + "="*50)

    with open(log_file_path, 'w', encoding='utf-8') as log_file:
        log_file.write("LOG RIWAYAT PEMBERSIHAN DATA (CLEANING LOG) - GOOGLE DRIVE\n")
        log_file.write("="*50 + "\n")

        success_count = 0

        for idx, filename in enumerate(sorted(pdf_files), 1):
            pdf_path = os.path.join(SOURCE_PDF_DIR, filename)
            case_id = f"case_{idx:03d}"
            output_txt_path = os.path.join(RAW_DIR, f"{case_id}.txt")

            try:
                # 1. Ekstraksi dari Drive
                raw_text = extract_text(pdf_path)
                initial_length = len(raw_text)

                # 2. Bersihkan teks
                cleaned_text = clean_indonesian_court_text(raw_text)
                final_length = len(cleaned_text)

                # 3. Validasi Keutuhan (minimal 80% teks tersedia / rasio masuk akal)
                if final_length > 0 and (final_length / initial_length) >= 0.5:
                    status = "VALID"

                    # Menyimpan langsung ke Drive
                    with open(output_txt_path, 'w', encoding='utf-8') as txt_file:
                        txt_file.write(cleaned_text)
                    success_count += 1
                else:
                    status = "WARNING (Teks terlalu pendek/terpotong)"

                log_msg = f"[{case_id}] File: {filename} | Karakter Awal: {initial_length} | Pasca-Bersih: {final_length} | Status: {status}\n"
                log_file.write(log_msg)
                print(f"Berhasil memproses [{case_id}] -> {filename}")

            except Exception as e:
                error_msg = f"[{case_id}] GAGAL memproses {filename}. Error: {str(e)}\n"
                log_file.write(error_msg)
                print(error_msg.strip())

        print("="*50)
        print(f"Proses Selesai! {success_count} file teks bersih tersimpan aman di Google Drive-mu ('/data/raw/')")
        print(f"File log dapat dilihat di Drive: {log_file_path}")

Menemukan 35 file PDF di Google Drive. Memulai proses...
Berhasil memproses [case_001] -> putusan_117_pdt_2023_pt_sby_20260610063252.pdf
Berhasil memproses [case_002] -> putusan_147_pdt_2026_pt_sby_20260610063722.pdf
Berhasil memproses [case_003] -> putusan_172_pdt_2024_pt_sby_20260610063357.pdf
Berhasil memproses [case_004] -> putusan_174_pdt_2026_pt_sby_20260610064022.pdf
Berhasil memproses [case_005] -> putusan_184_pdt_2021_pt_sby_20260610063312.pdf
Berhasil memproses [case_006] -> putusan_18_pdt_2023_pt_sby_20260610063623.pdf
Berhasil memproses [case_007] -> putusan_240_pdt_2026_pt_sby_20260610064013.pdf
Berhasil memproses [case_008] -> putusan_265_pdt_2020_pt_sby_20260610062916.pdf
Berhasil memproses [case_009] -> putusan_276_pdt_2023_pt_sby_20260610063321.pdf
Berhasil memproses [case_010] -> putusan_296_pdt_2021_pt_sby_20260610063056.pdf
Berhasil memproses [case_011] -> putusan_312_pdt_2018_pt_sby_20260610063823.pdf
Berhasil memproses [case_012] -> putusan_312_pdt_2020_pt_sby_202